# HookLink2iOS — IPA válida do Actions → Colab → Drive

Transferências em chunks de 16 MiB, retomada HTTP Range, SHA-256 e readback Drive.
Último build original #18 (run 43429056395, 13/11/2026) deu sucesso, mas seu artefato retornava zero arquivos em 22/11/2026. Se necessário, configure backup Drive ou arquivo local; não recompila nem escolhe outra branch.

**Fonte original confirmada no Drive:** `1cd85ZhbrJXZo5VK1GMQPxA-8R17qpPPD` (3.476.939.759 bytes). O notebook prioriza a última IPA de run bem-sucedido com artefato existente; esta fonte é o fallback.


In [ ]:
# HookLink2iOS: última IPA legítima, Actions → Colab → Drive, 16 MiB por chunk
REPO="joao9vegeto-stack/KintTany"
BRANCH="HookLink2iOS"
TITLE="0400486024DEC000"
PROJECT_DRIVE_FOLDER="1IynIGg57WnKydeqsdVzOp90tPkiX_eNY"
SOURCE_DRIVE_IPA_ID="1cd85ZhbrJXZo5VK1GMQPxA-8R17qpPPD" # IPA original validada: 3.476.939.759 bytes
SOURCE_LOCAL_IPA=""    # Alternativa: /content/arquivo.ipa ou ZIP do artefato
CHUNK=16*1024*1024
import os,time,re,json,io,hashlib,zipfile,plistlib,shutil,getpass
from pathlib import Path
from urllib.parse import urlparse
import requests
from google.colab import auth,userdata
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload,MediaFileUpload
import google.auth
ROOT=Path("/content/HookLink2iOS-transfer");ROOT.mkdir(exist_ok=True)
try:TOKEN=userdata.get("GH_TOKEN") or ""
except Exception:TOKEN=""
if not TOKEN:TOKEN=getpass.getpass("GH_TOKEN (Actions:read para localizar a última IPA): ").strip()
if not TOKEN:raise RuntimeError("GH_TOKEN é obrigatório para localizar artefatos neste repositório")
HEAD={"Accept":"application/vnd.github+json","X-GitHub-Api-Version":"2022-11-28"}
def gitget(path,**opts):
    headers=dict(HEAD)
    if TOKEN:headers["Authorization"]="Bearer "+TOKEN
    r=requests.get("https://api.github.com/repos/"+REPO+path,headers=headers,timeout=40,**opts)
    if r.status_code in (401,403):raise RuntimeError("Configure Colab Secrets GH_TOKEN com permissão Actions read.")
    r.raise_for_status();return r.json()
def sha(path,algo="sha256"):
    h=hashlib.new(algo)
    with open(path,"rb") as f:
        for b in iter(lambda:f.read(CHUNK),b""):h.update(b)
    return h.hexdigest()
auth.authenticate_user()
creds,_=google.auth.default(scopes=["https://www.googleapis.com/auth/drive"])
drive=build("drive","v3",credentials=creds,cache_discovery=False)
runs=gitget("/actions/runs",params={"branch":BRANCH,"status":"success","per_page":100})["workflow_runs"]
okpaths={".github/workflows/hooklink2ios.yml",".github/workflows/hooklink2ios-perf-repack.yml"}
runs=[r for r in runs if r.get("head_branch")==BRANCH and r.get("path") in okpaths and r.get("conclusion")=="success"]
runs.sort(key=lambda r:r["created_at"],reverse=True)
artifact=None;chosenrun=None
for run in runs:
    opts=gitget(f"/actions/runs/{run['id']}/artifacts").get("artifacts",[])
    opts=[a for a in opts if not a.get("expired") and "HookLink2iOS".lower() in a.get("name","").lower() and a.get("size_in_bytes",0)>20*1024*1024]
    print(f"run {run['id']} #{run['run_number']}: {len(opts)} IPA artifacts available")
    if opts:artifact=opts[0];chosenrun=run;break
if not artifact:print("A API não retornou IPA disponível. O último build original #18 foi sucesso em 13/11, mas seu artifact expirou/foi removido. Use SOURCE_DRIVE_IPA_ID ou SOURCE_LOCAL_IPA.")


In [ ]:
# Download Actions ranged 16 MiB, resume from existing partial bytes, verify each Content-Range.
def download_actions(art):
    global TOKEN
    if not TOKEN:TOKEN=getpass.getpass("GH_TOKEN (Actions read): ").strip()
    meta=gitget("/actions/artifacts/"+str(art["id"]))
    if meta.get("expired"):raise RuntimeError("Artifact expirado. Não há arquivo para baixar.")
    total=int(meta["size_in_bytes"])
    partial=ROOT/f"artifact-{art['id']}.zip.part"
    finished=ROOT/f"artifact-{art['id']}.zip"
    marker=ROOT/f"artifact-{art['id']}.meta.json"
    identity={"id":art["id"],"bytes":total,"digest":meta.get("digest")}
    if not marker.exists() or json.loads(marker.read_text())!=identity:
        partial.unlink(missing_ok=True);marker.write_text(json.dumps(identity))
    pos=partial.stat().st_size if partial.exists() else 0
    if pos>total:partial.unlink();pos=0
    while pos<total:
        end=min(total,pos+CHUNK)-1
        for attempt in range(8):
            try:
                headers={**HEAD,"Authorization":"Bearer "+TOKEN}
                rr=requests.get(f"https://api.github.com/repos/{REPO}/actions/artifacts/{art['id']}/zip",headers=headers,allow_redirects=False,timeout=40)
                if rr.status_code!=302 or not rr.headers.get("Location"):raise RuntimeError(f"GitHub redirect HTTP {rr.status_code}")
                signed=rr.headers["Location"]
                if urlparse(signed).scheme!="https":raise RuntimeError("Redirect inseguro")
                with requests.get(signed,headers={"Range":f"bytes={pos}-{end}"},stream=True,timeout=(30,180)) as r:
                    m=re.fullmatch(r"bytes\s+(\d+)-(\d+)/(\d+)",r.headers.get("Content-Range",""))
                    if r.status_code!=206 or not m or tuple(map(int,m.groups()))!=(pos,end,total):raise RuntimeError("HTTP Range inválido")
                    received=0
                    with partial.open("r+b" if partial.exists() else "w+b") as f:
                        f.truncate(pos);f.seek(pos)
                        for b in r.iter_content(1024*1024):
                            if b:f.write(b);received+=len(b)
                    if received!=end-pos+1:raise RuntimeError("Chunk incompleto")
                pos=end+1
                print(f"Actions → Colab: {pos/total*100:.1f}% ({pos:,}/{total:,})",end="\r")
                break
            except Exception as e:
                if partial.exists():
                    with partial.open("r+b") as f:f.truncate(pos)
                print("\nChunk retry:",type(e).__name__,str(e))
                if attempt==7:raise
                time.sleep(min(30,2**attempt))
    print()
    partial.replace(finished)
    dg=meta.get("digest") or ""
    if dg.startswith("sha256:") and sha(finished)!=dg.split(":",1)[1]:raise RuntimeError("SHA-256 do artefato GitHub não confere")
    return finished
def get_drive_backup(fid):
    info=drive.files().get(fileId=fid,fields="name,size,md5Checksum",supportsAllDrives=True).execute()
    out=ROOT/"source-drive.bin"
    with out.open("wb") as f:
        loader=MediaIoBaseDownload(f,drive.files().get_media(fileId=fid,supportsAllDrives=True),chunksize=CHUNK)
        done=False
        while not done:
            for attempt in range(8):
                try:
                    status,done=loader.next_chunk()
                    print(f"Drive → Colab: {status.progress()*100:.1f}%",end="\r")
                    break
                except Exception:
                    if attempt==7:raise
                    time.sleep(min(30,2**attempt))
    print()
    if int(info["size"])!=out.stat().st_size or (info.get("md5Checksum") and sha(out,"md5")!=info["md5Checksum"]):raise RuntimeError("Fonte backup Drive não confere")
    return out
if artifact:
    SOURCE=download_actions(artifact)
    origin=f"Actions run {chosenrun['id']} artifact {artifact['id']}"
elif SOURCE_DRIVE_IPA_ID.strip():
    SOURCE=get_drive_backup(SOURCE_DRIVE_IPA_ID.strip());origin="Cópia privada Drive"
elif SOURCE_LOCAL_IPA.strip() and Path(SOURCE_LOCAL_IPA).is_file():
    SOURCE=Path(SOURCE_LOCAL_IPA);origin="Arquivo Colab"
else:
    raise FileNotFoundError("Última IPA original indisponível no Actions. Configure SOURCE_DRIVE_IPA_ID ou SOURCE_LOCAL_IPA com uma cópia existente; o notebook NÃO recompila o jogo.")
with zipfile.ZipFile(SOURCE) as z:
    nested=[i for i in z.infolist() if i.filename.lower().endswith(".ipa") and not i.is_dir()]
    if nested:
        if len(nested)!=1:raise RuntimeError("Mais de uma IPA no artefato")
        IPA=ROOT/Path(nested[0].filename).name
        with z.open(nested[0]) as src,IPA.open("wb") as dest:shutil.copyfileobj(src,dest,CHUNK)
        reports=[i for i in z.infolist() if Path(i.filename).name=="build-report.json"]
        if len(reports)==1:
            report=json.loads(z.read(reports[0]))
            want=report.get("ipa",{}).get("sha256")
            if want and sha(IPA)!=want:raise RuntimeError("SHA-256 divergente do build-report")
    else:IPA=SOURCE
with zipfile.ZipFile(IPA) as z:
    names=z.namelist();pre="Payload/DmIOS.app/"
    if len(names)!=len(set(names)) or any(pre+x not in names for x in ("DmIOS","romfs.id","Info.plist")):raise RuntimeError("IPA incompleta")
    marker=z.read(pre+"romfs.id").decode("utf8","replace")
    if "title="+TITLE not in marker:raise RuntimeError("Title ID incorreto")
    if str(plistlib.loads(z.read(pre+"Info.plist")).get("MinimumOSVersion"))!="26.0":raise RuntimeError("iOS target incorreto")
    if not any(n.startswith(pre+"msl/b-") and n.endswith(".img") for n in names):raise RuntimeError("Metal image ausente")
    if any(n.lower().endswith("romfs.img") for n in names):raise RuntimeError("RomFS deveria ser externo")
    if z.testzip():raise RuntimeError("ZIP CRC inválido")
IPA_HASH=sha(IPA)
print("IPA verificada:",IPA.name,IPA.stat().st_size,"bytes, SHA-256:",IPA_HASH)


In [ ]:
# Upload Colab → Drive resumable 16 MiB + MD5/SHA-256 readback.
def drive_folder():
    parent=PROJECT_DRIVE_FOLDER
    try:drive.files().get(fileId=parent,fields="id",supportsAllDrives=True).execute()
    except Exception:
        found=drive.files().list(q="name='HookLink2iOS' and mimeType='application/vnd.google-apps.folder' and trashed=false",fields="files(id)",pageSize=20).execute().get("files",[])
        parent=found[0]["id"] if found else drive.files().create(body={"name":"HookLink2iOS","mimeType":"application/vnd.google-apps.folder"},fields="id").execute()["id"]
    found=drive.files().list(q=f"'{parent}' in parents and name='ipa' and mimeType='application/vnd.google-apps.folder' and trashed=false",fields="files(id)",pageSize=20).execute().get("files",[])
    return found[0]["id"] if found else drive.files().create(body={"name":"ipa","mimeType":"application/vnd.google-apps.folder","parents":[parent]},fields="id").execute()["id"]
folder=drive_folder()
name=IPA.stem+"-"+IPA_HASH[:12]+".ipa"
queryname=name.replace("'","\\'")
existing=drive.files().list(q=f"'{folder}' in parents and name='{queryname}' and trashed=false",fields="files(id,name,size,md5Checksum,appProperties,webViewLink)",pageSize=100).execute().get("files",[])
MD5=sha(IPA,"md5")
current=next((x for x in existing if int(x.get("size",0))==IPA.stat().st_size and x.get("md5Checksum")==MD5 and (x.get("appProperties") or {}).get("sha256")==IPA_HASH),None)
if current:
    print("Cópia idêntica encontrada no Drive; upload desnecessário.")
else:
    upload=MediaFileUpload(str(IPA),mimetype="application/octet-stream",chunksize=CHUNK,resumable=True)
    req=drive.files().create(body={"name":name,"parents":[folder],"appProperties":{"sha256":IPA_HASH,"titleId":TITLE,"branch":BRANCH}},media_body=upload,fields="id,name,size,md5Checksum,appProperties,webViewLink")
    current=None
    while current is None:
        for attempt in range(8):
            try:
                status,current=req.next_chunk()
                if status:print(f"Colab → Drive: {status.progress()*100:.1f}%",end="\r")
                break
            except Exception as e:
                print("\nRetry upload chunk:",type(e).__name__)
                if attempt==7:raise
                time.sleep(min(30,2**attempt))
    print()
check=drive.files().get(fileId=current["id"],fields="id,name,size,md5Checksum,appProperties,webViewLink",supportsAllDrives=True).execute()
if int(check.get("size",0))!=IPA.stat().st_size or check.get("md5Checksum")!=MD5 or (check.get("appProperties") or {}).get("sha256")!=IPA_HASH:
    raise RuntimeError("Drive readback não validou tamanho, MD5 e SHA-256")
print("TRANSFERÊNCIA INTEGRALMENTE VALIDADA")
print("Origem:",origin)
print("Nome:",check["name"])
print("SHA-256:",IPA_HASH)
print("Drive:",check.get("webViewLink") or "https://drive.google.com/file/d/"+check["id"]+"/view")
